# Lab | Data Cleaning and Formatting

In this lab, we will be working with the customer data from an insurance company, which can be found in the CSV file located at the following link: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv


# Challenge 1: Data Cleaning and Formatting

## Exercise 1: Cleaning Column Names

To ensure consistency and ease of use, standardize the column names of the dataframe. Start by taking a first look at the dataframe and identifying any column names that need to be modified. Use appropriate naming conventions and make sure that column names are descriptive and informative.

*Hint*:
- *Column names should be in lower case*
- *White spaces in column names should be replaced by `_`*
- *`st` could be replaced for `state`*

In [2]:
# Your code here
import pandas as pd

# The URL for the file (file1.csv is the current working context)
data_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"

try:
    # 1. LOAD THE DATA
    # We must successfully load the data first to access the column names.
    df = pd.read_csv(data_url)
    print("Data loaded successfully.")
    
    # Check the original column names
    print("\n--- Original Column Names ---")
    print(df.columns.tolist())
    
    # 2. CLEANING FUNCTION
    def clean_column_name(col_name):
        """Standardizes column name to lowercase, replaces spaces with underscores, 
           and handles the 'ST' to 'state' conversion."""
        
        # 1. Lowercase and replace spaces
        new_name = col_name.lower().replace(' ', '_')
        
        # 2. Handle the specific 'st' case (assuming 'ST' becomes 'st' then we rename it)
        if new_name == 'st':
            return 'state'
        
        # 3. Handle 'customer_lifetime_value' (often has extra chars, but let's stick to spaces)
        # We will also remove trailing/leading whitespace and symbols that pandas might pick up
        new_name = new_name.replace('-', '_').replace('(', '').replace(')', '').strip('_')

        return new_name

    # 3. Apply the cleaning function to all columns
    df.columns = [clean_column_name(col) for col in df.columns]
    
    # 4. Display the results
    print("\n--- Cleaned Column Names ---")
    print(df.columns.tolist())
    
    print("\nColumn name cleaning complete. Displaying the first 5 rows with new names:")
    print(df.head().to_markdown(index=False))

except Exception as e:
    print(f"\nFATAL ERROR: Data loading failed. Check your network or URL. Error: {e}")

Data loaded successfully.

--- Original Column Names ---
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']

--- Cleaned Column Names ---
['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']

Column name cleaning complete. Displaying the first 5 rows with new names:
| customer   | state      | gender   | education            | customer_lifetime_value   |   income |   monthly_premium_auto | number_of_open_complaints   | policy_type    | vehicle_class   |   total_claim_amount |
|:-----------|:-----------|:---------|:---------------------|:--------------------------|---------:|-----------------------:|:----------------------------|:---------------|:----------------|---------------------:|
| RB50392    | Washington | nan   

## Exercise 2: Cleaning invalid Values

The dataset contains columns with inconsistent and incorrect values that could affect the accuracy of our analysis. Therefore, we need to clean these columns to ensure that they only contain valid data.

Note that this exercise will focus only on cleaning inconsistent values and will not involve handling null values (NaN or None).

*Hint*:
- *Gender column contains various inconsistent values such as "F", "M", "Femal", "Male", "female", which need to be standardized, for example, to "M" and "F".*
- *State abbreviations be can replaced with its full name, for example "AZ": "Arizona", "Cali": "California", "WA": "Washington"*
- *In education, "Bachelors" could be replaced by "Bachelor"*
- *In Customer Lifetime Value, delete the `%` character*
- *In vehicle class, "Sports Car", "Luxury SUV" and "Luxury Car" could be replaced by "Luxury"*

In [3]:
# Your code here
import numpy as np

# Assuming df is loaded with clean column names (e.g., state, gender, clv)

# 1. Cleaning 'gender' Column
# Goal: Standardize to 'M' and 'F'
df['gender'] = df['gender'].str.upper().str.strip()
df['gender'] = df['gender'].replace({
    'FEMAL': 'F',
    'MALE': 'M',
    'FEMALE': 'F'
})
# After standardizing, we keep only 'M' and 'F'. Any other value is set to NaN/None
# (Though the instruction says not to handle nulls, this handles remaining invalid strings)
df.loc[~df['gender'].isin(['M', 'F']), 'gender'] = np.nan 

print("1. 'gender' column standardized to 'M' and 'F'.")


# 2. Cleaning 'state' Column (ST)
# Goal: Replace abbreviations with full names
state_mapping = {
    'AZ': 'Arizona',
    'Cali': 'California',
    'WA': 'Washington',
    # Note: 'state' was renamed from 'ST' and is likely upper-cased
    # We should also handle other common states like 'OR', 'NV' if they exist, 
    # but we stick to the provided mapping for the challenge.
}
# Apply the replacement; non-matched values remain unchanged.
df['state'] = df['state'].replace(state_mapping) 

print("2. 'state' abbreviations replaced with full names.")


# 3. Cleaning 'education' Column
# Goal: Replace "Bachelors" with "Bachelor"
df['education'] = df['education'].replace('Bachelors', 'Bachelor')

print("3. 'education' inconsistency fixed.")


# 4. Cleaning 'customer_lifetime_value' Column (Numerical)
# Goal: Delete the '%' character and ensure it is a float.
# Note: We assume the column name is 'customer_lifetime_value' after Exercise 1.
# The core numerical cleaning (removing '$' and ',') should have been done previously.
# This handles the specific '%' case and ensures final conversion to float.
df['customer_lifetime_value'] = (
    df['customer_lifetime_value']
    .astype(str)
    .str.replace(r'[%$]', '', regex=True) # Removing both % and $ just to be safe
    .str.strip()
    .astype(float)
)

print("4. '%' character removed from 'customer_lifetime_value' and converted to float.")


# 5. Cleaning 'vehicle_class' Column (Categorical Aggregation)
# Goal: Group similar luxury vehicles into a single 'Luxury' category
luxury_vehicles = ['Sports Car', 'Luxury SUV', 'Luxury Car']
df['vehicle_class'] = df['vehicle_class'].replace(luxury_vehicles, 'Luxury')

print("5. Luxury vehicle classes grouped into 'Luxury'.")


# --- Verification ---
print("\n--- Verification of Cleaned Data ---")
print("Gender unique counts:")
print(df['gender'].value_counts(dropna=False).to_markdown())

print("\nState unique counts (partial check):")
print(df['state'].value_counts(dropna=False).head().to_markdown())

print("\nVehicle Class unique counts:")
print(df['vehicle_class'].value_counts(dropna=False).to_markdown())

1. 'gender' column standardized to 'M' and 'F'.
2. 'state' abbreviations replaced with full names.
3. 'education' inconsistency fixed.
4. '%' character removed from 'customer_lifetime_value' and converted to float.
5. Luxury vehicle classes grouped into 'Luxury'.

--- Verification of Cleaned Data ---
Gender unique counts:
| gender   |   count |
|:---------|--------:|
| nan      |    3054 |
| F        |     502 |
| M        |     452 |

State unique counts (partial check):
| state      |   count |
|:-----------|--------:|
| nan        |    2937 |
| California |     331 |
| Oregon     |     320 |
| Arizona    |     211 |
| Washington |     111 |

Vehicle Class unique counts:
| vehicle_class   |   count |
|:----------------|--------:|
| nan             |    2937 |
| Four-Door Car   |     576 |
| Two-Door Car    |     205 |
| SUV             |     199 |
| Luxury          |      91 |


## Exercise 3: Formatting data types

The data types of many columns in the dataset appear to be incorrect. This could impact the accuracy of our analysis. To ensure accurate analysis, we need to correct the data types of these columns. Please update the data types of the columns as appropriate.

It is important to note that this exercise does not involve handling null values (NaN or None).

*Hint*:
- *Customer lifetime value should be numeric*
- *Number of open complaints has an incorrect format. Look at the different values it takes with `unique()` and take the middle value. As an example, 1/5/00 should be 5. Number of open complaints is a string - remember you can use `split()` to deal with it and take the number you need. Finally, since it should be numeric, cast the column to be in its proper type.*

In [6]:
# Your code here
import pandas as pd
import numpy as np

# --- Configuration ---
data_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
print("Starting Data Cleaning and Formatting...\n")

try:
    # --- 1. Data Loading ---
    df = pd.read_csv(data_url)
    print("Data loaded successfully.")

    # --- 2. Exercise 1: Cleaning Column Names ---
    def clean_column_name(col_name):
        """Standardizes column name to lowercase, replaces spaces with underscores, 
           and handles the 'ST' to 'state' conversion."""
        new_name = col_name.lower().replace(' ', '_')
        if new_name == 'st':
            return 'state'
        # Remove any leading/trailing underscores that might result from cleaning
        return new_name.strip('_')

    df.columns = [clean_column_name(col) for col in df.columns]
    print(f"1. Column names standardized. Example: 'ST' is now 'state'.")

    # --- 3. Exercise 2: Cleaning Inconsistent Values ---
    
    # a) Clean 'gender' column: Standardize to 'M' and 'F'
    df['gender'] = df['gender'].astype(str).str.upper().str.strip()
    df['gender'] = df['gender'].replace({
        'FEMAL': 'F',
        'MALE': 'M',
        'FEMALE': 'F'
    })
    # Replace non M/F values with NaN for consistency (though not strictly handling nulls)
    df.loc[~df['gender'].isin(['M', 'F']), 'gender'] = np.nan 

    # b) Clean 'state' column: Replace abbreviations with full names
    state_mapping = {
        'AZ': 'Arizona',
        'Cali': 'California',
        'WA': 'Washington'
    }
    df['state'] = df['state'].replace(state_mapping) 

    # c) Clean 'education' column
    df['education'] = df['education'].replace('Bachelors', 'Bachelor')

    # d) Clean 'vehicle_class' column: Group luxury vehicles
    luxury_vehicles = ['Sports Car', 'Luxury SUV', 'Luxury Car']
    df['vehicle_class'] = df['vehicle_class'].replace(luxury_vehicles, 'Luxury')
    
    print("2. Inconsistent values cleaned (Gender, State, Vehicle Class).")


    # --- 4. Exercise 3: Formatting Data Types (with Fix) ---
    
    # a) Correcting 'number_of_open_complaints' (Handling the 'X/Y/00' format)
    # FIX APPLIED: Use fillna(0) to prevent TypeError if a row splits into NoneType.
    df['number_of_open_complaints'] = (
        df['number_of_open_complaints']
        .astype(str)
        .str.split('/', expand=True)[1] # Extract the middle number
        .fillna(0) # Replace None values with 0
        .astype(int)
    )

    # b) Correcting General Numeric Columns (Ensuring correct type, using coerce for safety)
    df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')
    df['income'] = pd.to_numeric(df['income'], errors='coerce')
    df['total_claim_amount'] = pd.to_numeric(df['total_claim_amount'], errors='coerce')
    # Note: If income had '$' or ',' it would need explicit string replacement before to_numeric 
    # if errors='coerce' fails. Assuming it's relatively clean here.

    # c) Optimizing Categorical Columns (memory efficiency)
    categorical_cols = ['state', 'gender', 'education', 'policy_type', 'vehicle_class']
    for col in categorical_cols:
        df[col] = df[col].astype('category')
        
    print("3. Data types formatted (e.g., complaints fixed, categories optimized).")

    # --- Final Verification ---
    print("\n--- Final Cleaned DataFrame Info ---")
    df.info()

    print("\n--- Head of Cleaned Data ---")
    print(df.head().to_markdown(index=False))

except Exception as e:
    print(f"\nFATAL ERROR: Failed to execute the cleaning script. Error: {e}")

Starting Data Cleaning and Formatting...

Data loaded successfully.
1. Column names standardized. Example: 'ST' is now 'state'.
2. Inconsistent values cleaned (Gender, State, Vehicle Class).
3. Data types formatted (e.g., complaints fixed, categories optimized).

--- Final Cleaned DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4008 entries, 0 to 4007
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   customer                   1071 non-null   object  
 1   state                      1071 non-null   category
 2   gender                     954 non-null    category
 3   education                  1071 non-null   category
 4   customer_lifetime_value    0 non-null      float64 
 5   income                     1071 non-null   float64 
 6   monthly_premium_auto       1071 non-null   float64 
 7   number_of_open_complaints  4008 non-null   int64   
 8   policy_type      

## Exercise 4: Dealing with Null values

Identify any columns with null or missing values. Identify how many null values each column has. You can use the `isnull()` function in pandas to find columns with null values.

Decide on a strategy for handling the null values. There are several options, including:

- Drop the rows or columns with null values
- Fill the null values with a specific value (such as the column mean or median for numerical variables, and mode for categorical variables)
- Fill the null values with the previous or next value in the column
- Fill the null values based on a more complex algorithm or model (note: we haven't covered this yet)

Implement your chosen strategy to handle the null values. You can use the `fillna()` function in pandas to fill null values or `dropna()` function to drop null values.

Verify that your strategy has successfully handled the null values. You can use the `isnull()` function again to check if there are still null values in the dataset.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling null values.

After formatting data types, as a last step, convert all the numeric variables to integers.

In [12]:
# Your code here
import pandas as pd
import numpy as np

# Set pd.options.mode.chained_assignment = None to prevent SettingWithCopyWarning
# if using older versions of pandas, though using loc/df[] = ... should prevent it.

# --- Configuration ---
data_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
print("Starting Data Cleaning and Formatting...\n")

try:
    # --- 1. Data Loading ---
    df = pd.read_csv(data_url)
    print("Data loaded successfully.")

    # --- 2. Exercise 1: Cleaning Column Names ---
    def clean_column_name(col_name):
        """Standardizes column name to lowercase, replaces spaces with underscores, 
           and handles the 'ST' to 'state' conversion."""
        new_name = col_name.lower().replace(' ', '_')
        if new_name == 'st':
            return 'state'
        return new_name.strip('_')

    df.columns = [clean_column_name(col) for col in df.columns]
    print(f"1. Column names standardized. Example: 'ST' is now 'state'.")

    # --- 3. Exercise 2: Cleaning Inconsistent Values ---
    
    # a) Clean 'gender' column: Standardize to 'M' and 'F'
    df['gender'] = df['gender'].astype(str).str.upper().str.strip()
    df['gender'] = df['gender'].replace({
        'FEMAL': 'F',
        'MALE': 'M',
        'FEMALE': 'F'
    })
    # Replace non M/F values with NaN for consistency (to be handled in Ex 4)
    df.loc[~df['gender'].isin(['M', 'F']), 'gender'] = np.nan 

    # b) Clean 'state' column: Replace abbreviations with full names
    state_mapping = {
        'AZ': 'Arizona',
        'Cali': 'California',
        'WA': 'Washington'
    }
    df['state'] = df['state'].replace(state_mapping) 

    # c) Clean 'education' column
    df['education'] = df['education'].replace('Bachelors', 'Bachelor')

    # d) Clean 'vehicle_class' column: Group luxury vehicles
    luxury_vehicles = ['Sports Car', 'Luxury SUV', 'Luxury Car']
    df['vehicle_class'] = df['vehicle_class'].replace(luxury_vehicles, 'Luxury')
    
    print("2. Inconsistent values cleaned (Gender, State, Vehicle Class).")


    # --- 4. Exercise 3: Formatting Data Types (with Fix) ---
    
    # a) Correcting 'number_of_open_complaints' (Handling the 'X/Y/00' format)
    # FIX: Use fillna(0) to prevent TypeError during int conversion if split fails.
    df['number_of_open_complaints'] = (
        df['number_of_open_complaints']
        .astype(str)
        .str.split('/', expand=True)[1] # Extract the middle number
        .fillna(0) # Replace None values with 0
        .astype(int)
    )

    # b) Correcting General Numeric Columns (Casting to float, using coerce for safety)
    df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')
    df['income'] = pd.to_numeric(df['income'], errors='coerce')
    df['total_claim_amount'] = pd.to_numeric(df['total_claim_amount'], errors='coerce')

    # c) Optimizing Categorical Columns (memory efficiency)
    categorical_cols = ['state', 'gender', 'education', 'policy_type', 'vehicle_class']
    for col in categorical_cols:
        df[col] = df[col].astype('category')
        
    print("3. Data types formatted (complaints fixed, categories optimized).")


    # --- 5. Exercise 4: Dealing with Null Values ---
    print("\n--- 5. Exercise 4: Dealing with Null Values ---")

    # a) Identify columns with null values (Required step)
    print("\nInitial Null Count (before imputation):")
    null_counts = df.isnull().sum()
    print(null_counts[null_counts > 0].to_markdown())
    
    # Reasoning for Imputation Strategy:
    # 1. Numerical ('income'): Use the median as it is less sensitive to outliers than the mean.
    # 2. Categorical ('gender', 'education', etc.): Use the mode (most frequent category) 
    #    as it preserves the distribution's central tendency for non-ordinal categories.

    # b) Impute Numerical Nulls (Income)
    median_income = df['income'].median()
    df['income'].fillna(median_income, inplace=True)
    print(f"\nIncome nulls imputed using median income: £{median_income:,.2f}")

    # c) Impute Categorical Nulls (Gender, Education, EmploymentStatus, Response)
    for col in ['gender', 'education', 'employmentstatus', 'response']:
        if col in df.columns and df[col].isnull().any():
            # [0] selects the first mode in case of ties
            mode_val = df[col].mode()[0] 
            df[col].fillna(mode_val, inplace=True)
            print(f"{col} nulls imputed using mode: {mode_val}")


    # d) Verification (Required step)
    print("\nFinal Null Count (Only showing columns with > 0 nulls):")
    final_null_counts = df.isnull().sum()
    print(final_null_counts[final_null_counts > 0].to_markdown())


    # --- 6. Final Numeric Conversion to Integer (Last step per instruction) ---
    print("\n--- 6. Final Numeric Conversion ---")
    
    # List of float columns to be converted to integer type
    # We use 'Int64' (Pandas Integer type) to handle potential remaining NaNs
    numeric_cols_to_int = ['customer_lifetime_value', 'income', 'monthly_premium_auto', 
                           'total_claim_amount', 'months_since_last_claim', 
                           'months_since_policy_inception', 'number_of_policies']
                           
    for col in numeric_cols_to_int:
        if col in df.columns and df[col].dtype != 'category':
            # Round to the nearest whole number before casting to int to prevent truncation
            df[col] = df[col].round(0).astype('Int64')
            
    print("All relevant numeric columns converted to integer type (Int64).")
    
    
    print("\n--- Data Cleaning and Formatting Complete! ---")

    # --- Final Verification ---
    print("\n--- Final Cleaned DataFrame Info ---")
    df.info()

    print("\n--- Head of Cleaned Data ---")
    print(df.head().to_markdown(index=False))

except Exception as e:
    print(f"\nFATAL ERROR: Failed to execute the cleaning script. Error: {e}")

Starting Data Cleaning and Formatting...

Data loaded successfully.
1. Column names standardized. Example: 'ST' is now 'state'.
2. Inconsistent values cleaned (Gender, State, Vehicle Class).
3. Data types formatted (complaints fixed, categories optimized).

--- 5. Exercise 4: Dealing with Null Values ---

Initial Null Count (before imputation):
|                         |    0 |
|:------------------------|-----:|
| customer                | 2937 |
| state                   | 2937 |
| gender                  | 3054 |
| education               | 2937 |
| customer_lifetime_value | 4008 |
| income                  | 2937 |
| monthly_premium_auto    | 2937 |
| policy_type             | 2937 |
| vehicle_class           | 2937 |
| total_claim_amount      | 2937 |

Income nulls imputed using median income: £36,234.00
gender nulls imputed using mode: F
education nulls imputed using mode: Bachelor

Final Null Count (Only showing columns with > 0 nulls):
|                         |    0 |
|:-----

C:\Users\safin\AppData\Local\Temp\ipykernel_39700\1963415753.py:99: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['income'].fillna(median_income, inplace=True)
C:\Users\safin\AppData\Local\Temp\ipykernel_39700\1963415753.py:107: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For 

## Exercise 5: Dealing with duplicates

Use the `.duplicated()` method to identify any duplicate rows in the dataframe.

Decide on a strategy for handling the duplicates. Options include:
- Dropping all duplicate rows
- Keeping only the first occurrence of each duplicated row
- Keeping only the last occurrence of each duplicated row
- Dropping duplicates based on a subset of columns
- Dropping duplicates based on a specific column

Implement your chosen strategy using the `drop_duplicates()` function.

Verify that your strategy has successfully handled the duplicates by checking for duplicates again using `.duplicated()`.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling duplicates.

Save the cleaned dataset to a new CSV file.

*Hint*: *after dropping duplicates, reset the index to ensure consistency*.

In [8]:
# Your code here
import pandas as pd
import numpy as np

# Set pd.options.mode.chained_assignment = None to prevent SettingWithCopyWarning
# if using older versions of pandas, though using loc/df[] = ... should prevent it.

# --- Configuration ---
data_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
print("Starting Data Cleaning and Formatting...\n")

try:
    # --- 1. Data Loading ---
    df = pd.read_csv(data_url)
    print("Data loaded successfully.")

    # --- 2. Exercise 1: Cleaning Column Names ---
    def clean_column_name(col_name):
        """Standardizes column name to lowercase, replaces spaces with underscores, 
           and handles the 'ST' to 'state' conversion."""
        new_name = col_name.lower().replace(' ', '_')
        if new_name == 'st':
            return 'state'
        return new_name.strip('_')

    df.columns = [clean_column_name(col) for col in df.columns]
    print(f"1. Column names standardized. Example: 'ST' is now 'state'.")

    # --- 3. Exercise 2: Cleaning Inconsistent Values ---
    
    # a) Clean 'gender' column: Standardize to 'M' and 'F'
    df['gender'] = df['gender'].astype(str).str.upper().str.strip()
    df['gender'] = df['gender'].replace({
        'FEMAL': 'F',
        'MALE': 'M',
        'FEMALE': 'F'
    })
    # Replace non M/F values with NaN for consistency (to be handled in Ex 4)
    df.loc[~df['gender'].isin(['M', 'F']), 'gender'] = np.nan 

    # b) Clean 'state' column: Replace abbreviations with full names
    state_mapping = {
        'AZ': 'Arizona',
        'Cali': 'California',
        'WA': 'Washington'
    }
    df['state'] = df['state'].replace(state_mapping) 

    # c) Clean 'education' column
    df['education'] = df['education'].replace('Bachelors', 'Bachelor')

    # d) Clean 'vehicle_class' column: Group luxury vehicles
    luxury_vehicles = ['Sports Car', 'Luxury SUV', 'Luxury Car']
    df['vehicle_class'] = df['vehicle_class'].replace(luxury_vehicles, 'Luxury')
    
    print("2. Inconsistent values cleaned (Gender, State, Vehicle Class).")


    # --- 4. Exercise 3: Formatting Data Types (with Fix) ---
    
    # a) Correcting 'number_of_open_complaints' (Handling the 'X/Y/00' format)
    # FIX: Use fillna(0) to prevent TypeError during int conversion if split fails.
    df['number_of_open_complaints'] = (
        df['number_of_open_complaints']
        .astype(str)
        .str.split('/', expand=True)[1] # Extract the middle number
        .fillna(0) # Replace None values with 0
        .astype(int)
    )

    # b) Correcting General Numeric Columns (Casting to float, using coerce for safety)
    df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')
    df['income'] = pd.to_numeric(df['income'], errors='coerce')
    df['total_claim_amount'] = pd.to_numeric(df['total_claim_amount'], errors='coerce')

    # c) Optimizing Categorical Columns (memory efficiency)
    categorical_cols = ['state', 'gender', 'education', 'policy_type', 'vehicle_class']
    for col in categorical_cols:
        df[col] = df[col].astype('category')
        
    print("3. Data types formatted (complaints fixed, categories optimized).")


    # --- 5. Exercise 4: Dealing with Null Values ---
    print("\n--- 5. Exercise 4: Dealing with Null Values ---")

    # a) Identify columns with null values (Required step)
    print("\nInitial Null Count (before imputation):")
    null_counts = df.isnull().sum()
    print(null_counts[null_counts > 0].to_markdown())
    
    # Reasoning for Imputation Strategy:
    # 1. Numerical ('income'): Use the median as it is less sensitive to outliers than the mean.
    # 2. Categorical ('gender', 'education', etc.): Use the mode (most frequent category) 
    #    as it preserves the distribution's central tendency for non-ordinal categories.

    # b) Impute Numerical Nulls (Income)
    median_income = df['income'].median()
    df['income'].fillna(median_income, inplace=True)
    print(f"\nIncome nulls imputed using median income: £{median_income:,.2f}")

    # c) Impute Categorical Nulls (Gender, Education, EmploymentStatus, Response)
    for col in ['gender', 'education', 'employmentstatus', 'response']:
        if col in df.columns and df[col].isnull().any():
            # [0] selects the first mode in case of ties
            mode_val = df[col].mode()[0] 
            df[col].fillna(mode_val, inplace=True)
            print(f"{col} nulls imputed using mode: {mode_val}")


    # d) Verification (Required step)
    print("\nFinal Null Count (Only showing columns with > 0 nulls):")
    final_null_counts = df.isnull().sum()
    print(final_null_counts[final_null_counts > 0].to_markdown())


    # --- 6. Final Numeric Conversion to Integer (Last step per instruction) ---
    print("\n--- 6. Final Numeric Conversion ---")
    
    # List of float columns to be converted to integer type
    # We use 'Int64' (Pandas Integer type) to handle potential remaining NaNs
    numeric_cols_to_int = ['customer_lifetime_value', 'income', 'monthly_premium_auto', 
                           'total_claim_amount', 'months_since_last_claim', 
                           'months_since_policy_inception', 'number_of_policies']
                           
    for col in numeric_cols_to_int:
        if col in df.columns and df[col].dtype != 'category':
            # Round to the nearest whole number before casting to int to prevent truncation
            df[col] = df[col].round(0).astype('Int64')
            
    print("All relevant numeric columns converted to integer type (Int64).")
    
    
    # --- 7. Exercise 5: Dealing with Duplicates ---
    print("\n--- 7. Exercise 5: Dealing with Duplicates and Final Save ---")

    # a) Identify and count duplicates
    initial_duplicates = df.duplicated().sum()
    print(f"Initial count of duplicate rows: {initial_duplicates}")

    # b) Strategy: Drop duplicates, keeping the first occurrence.
    # Reasoning: This assumes the first record is the most complete or primary entry.
    df.drop_duplicates(keep='first', inplace=True)
    print("Strategy: Dropped all duplicate rows, keeping the first occurrence.")

    # c) Verification
    final_duplicates = df.duplicated().sum()
    print(f"Final count of duplicate rows (Verification): {final_duplicates}")

    # d) Reset Index (Hint)
    df.reset_index(drop=True, inplace=True)
    print("DataFrame index reset for consistency.")

    # e) Save the cleaned dataset to a new CSV file
    df.to_csv('cleaned_insurance_data.csv', index=False)
    print("\nSuccessfully saved the cleaned DataFrame to 'cleaned_insurance_data.csv'.")
    
    
    print("\n--- Data Cleaning and Formatting Complete! ---")

    # --- Final Verification ---
    print("\n--- Final Cleaned DataFrame Info ---")
    df.info()

    print("\n--- Head of Cleaned Data ---")
    print(df.head().to_markdown(index=False))

except Exception as e:
    print(f"\nFATAL ERROR: Failed to execute the cleaning script. Error: {e}")

Starting Data Cleaning and Formatting...

Data loaded successfully.
1. Column names standardized. Example: 'ST' is now 'state'.
2. Inconsistent values cleaned (Gender, State, Vehicle Class).
3. Data types formatted (complaints fixed, categories optimized).

--- 5. Exercise 4: Dealing with Null Values ---

Initial Null Count (before imputation):
|                         |    0 |
|:------------------------|-----:|
| customer                | 2937 |
| state                   | 2937 |
| gender                  | 3054 |
| education               | 2937 |
| customer_lifetime_value | 4008 |
| income                  | 2937 |
| monthly_premium_auto    | 2937 |
| policy_type             | 2937 |
| vehicle_class           | 2937 |
| total_claim_amount      | 2937 |

Income nulls imputed using median income: £36,234.00
gender nulls imputed using mode: F
education nulls imputed using mode: Bachelor

Final Null Count (Only showing columns with > 0 nulls):
|                         |    0 |
|:-----

C:\Users\safin\AppData\Local\Temp\ipykernel_39700\3142118966.py:99: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['income'].fillna(median_income, inplace=True)
C:\Users\safin\AppData\Local\Temp\ipykernel_39700\3142118966.py:107: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For 

# Bonus: Challenge 2: creating functions on a separate `py` file

Put all the data cleaning and formatting steps into functions, and create a main function that performs all the cleaning and formatting.

Write these functions in separate .py file(s). By putting these steps into functions, we can make the code more modular and easier to maintain.

*Hint: autoreload module is a utility module in Python that allows you to automatically reload modules in the current session when changes are made to the source code. This can be useful in situations where you are actively developing code and want to see the effects of changes you make without having to constantly restart the Python interpreter or Jupyter Notebook kernel.*

In [9]:
# Your code here
import pandas as pd
import numpy as np

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardizes column names by converting to lowercase, replacing spaces with 
    underscores, and handling specific renaming (e.g., 'ST' to 'state').

    Args:
        df: The input Pandas DataFrame.

    Returns:
        The DataFrame with standardized column names.
    """
    def standardize_name(col_name):
        new_name = col_name.lower().replace(' ', '_')
        if new_name == 'st':
            return 'state'
        # Remove any leading/trailing underscores that might result from cleaning
        return new_name.strip('_')

    df.columns = [standardize_name(col) for col in df.columns]
    return df

def clean_and_format_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Performs comprehensive data cleaning, formatting, null value handling, 
    and duplicate removal (Exercises 2, 3, 4, 5).

    Note: This function assumes column names have already been cleaned.

    Args:
        df: The input Pandas DataFrame with standardized column names.

    Returns:
        The fully cleaned and formatted DataFrame.
    """
    
    # 1. Cleaning Inconsistent Values (Exercise 2)
    
    # a) Gender: Standardize to 'M' and 'F'
    df['gender'] = df['gender'].astype(str).str.upper().str.strip()
    df['gender'] = df['gender'].replace({'FEMAL': 'F', 'MALE': 'M', 'FEMALE': 'F'})
    df.loc[~df['gender'].isin(['M', 'F']), 'gender'] = np.nan # Mark inconsistent as NaN

    # b) State: Replace abbreviations
    state_mapping = {'AZ': 'Arizona', 'Cali': 'California', 'WA': 'Washington'}
    df['state'] = df['state'].replace(state_mapping) 

    # c) Education: Fix 'Bachelors'
    df['education'] = df['education'].replace('Bachelors', 'Bachelor')

    # d) Vehicle Class: Group luxury vehicles
    luxury_vehicles = ['Sports Car', 'Luxury SUV', 'Luxury Car']
    df['vehicle_class'] = df['vehicle_class'].replace(luxury_vehicles, 'Luxury')
    
    
    # 2. Formatting Data Types (Exercise 3)
    
    # a) Fix 'number_of_open_complaints' format
    df['number_of_open_complaints'] = (
        df['number_of_open_complaints']
        .astype(str)
        .str.split('/', expand=True)[1]
        .fillna(0) # Handle NoneType resulting from split
        .astype(int)
    )

    # b) Correct General Numeric Columns (Casting to float first for safety)
    df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')
    df['income'] = pd.to_numeric(df['income'], errors='coerce')
    df['total_claim_amount'] = pd.to_numeric(df['total_claim_amount'], errors='coerce')

    # c) Optimize Categorical Columns
    categorical_cols = ['state', 'gender', 'education', 'policy_type', 'vehicle_class']
    for col in categorical_cols:
        if col in df.columns:
             df[col] = df[col].astype('category')
             

    # 3. Dealing with Null Values (Exercise 4)

    # a) Impute Numerical Nulls ('income'): using the median
    median_income = df['income'].median()
    df['income'].fillna(median_income, inplace=True)

    # b) Impute Categorical Nulls (Gender, Education, EmploymentStatus, Response): using the mode
    for col in ['gender', 'education', 'employmentstatus', 'response']:
        if col in df.columns and df[col].isnull().any():
            mode_val = df[col].mode()[0] 
            df[col].fillna(mode_val, inplace=True)
            
            
    # 4. Final Numeric Conversion (Exercise 4 - last step)

    numeric_cols_to_int = ['customer_lifetime_value', 'income', 'monthly_premium_auto', 
                           'total_claim_amount', 'months_since_last_claim', 
                           'months_since_policy_inception', 'number_of_policies']
                           
    for col in numeric_cols_to_int:
        if col in df.columns and df[col].dtype != 'category':
            df[col] = df[col].round(0).astype('Int64')

    
    # 5. Dealing with Duplicates (Exercise 5)
    
    # Drop duplicates, keeping the first occurrence
    df.drop_duplicates(keep='first', inplace=True)

    # Reset index
    df.reset_index(drop=True, inplace=True)
    
    return df

# Bonus: Challenge 3: Analyzing Clean and Formated Data

You have been tasked with analyzing the data to identify potential areas for improving customer retention and profitability. Your goal is to identify customers with a high policy claim amount and a low customer lifetime value.

In the Pandas Lab, we only looked at high policy claim amounts because we couldn't look into low customer lifetime values. If we had tried to work with that column, we wouldn't have been able to because customer lifetime value wasn't clean and in its proper format. So after cleaning and formatting the data, let's get some more interesting insights!

Instructions:

- Review the statistics again for total claim amount and customer lifetime value to gain an understanding of the data.
- To identify potential areas for improving customer retention and profitability, we want to focus on customers with a high policy claim amount and a low customer lifetime value. Consider customers with a high policy claim amount to be those in the top 25% of the total claim amount, and clients with a low customer lifetime value to be those in the bottom 25% of the customer lifetime value. Create a pandas DataFrame object that contains information about customers with a policy claim amount greater than the 75th percentile and a customer lifetime value in the bottom 25th percentile.
- Use DataFrame methods to calculate summary statistics about the high policy claim amount and low customer lifetime value data. To do so, select both columns of the dataframe simultaneously and pass it to the `.describe()` method. This will give you descriptive statistics, such as mean, median, standard deviation, minimum and maximum values for both columns at the same time, allowing you to compare and analyze their characteristics.

In [11]:
# Your code here
import pandas as pd
import numpy as np

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardizes column names by converting to lowercase, replacing spaces with 
    underscores, and handling specific renaming (e.g., 'ST' to 'state').

    Args:
        df: The input Pandas DataFrame.

    Returns:
        The DataFrame with standardized column names.
    """
    def standardize_name(col_name):
        new_name = col_name.lower().replace(' ', '_')
        if new_name == 'st':
            return 'state'
        # Remove any leading/trailing underscores that might result from cleaning
        return new_name.strip('_')

    df.columns = [standardize_name(col) for col in df.columns]
    return df

def clean_and_format_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Performs comprehensive data cleaning, formatting, null value handling, 
    and duplicate removal (Exercises 2, 3, 4, 5).

    Note: This function assumes column names have already been cleaned.

    Args:
        df: The input Pandas DataFrame with standardized column names.

    Returns:
        The fully cleaned and formatted DataFrame.
    """
    
    # 1. Cleaning Inconsistent Values (Exercise 2)
    
    # a) Gender: Standardize to 'M' and 'F'
    df['gender'] = df['gender'].astype(str).str.upper().str.strip()
    df['gender'] = df['gender'].replace({'FEMAL': 'F', 'MALE': 'M', 'FEMALE': 'F'})
    df.loc[~df['gender'].isin(['M', 'F']), 'gender'] = np.nan # Mark inconsistent as NaN

    # b) State: Replace abbreviations
    state_mapping = {'AZ': 'Arizona', 'Cali': 'California', 'WA': 'Washington'}
    df['state'] = df['state'].replace(state_mapping) 

    # c) Education: Fix 'Bachelors'
    df['education'] = df['education'].replace('Bachelors', 'Bachelor')

    # d) Vehicle Class: Group luxury vehicles
    luxury_vehicles = ['Sports Car', 'Luxury SUV', 'Luxury Car']
    df['vehicle_class'] = df['vehicle_class'].replace(luxury_vehicles, 'Luxury')
    
    
    # 2. Formatting Data Types (Exercise 3)
    
    # a) Fix 'number_of_open_complaints' format
    df['number_of_open_complaints'] = (
        df['number_of_open_complaints']
        .astype(str)
        .str.split('/', expand=True)[1]
        .fillna(0) # Handle NoneType resulting from split
        .astype(int)
    )

    # b) Correct General Numeric Columns (Casting to float first for safety)
    df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')
    df['income'] = pd.to_numeric(df['income'], errors='coerce')
    df['total_claim_amount'] = pd.to_numeric(df['total_claim_amount'], errors='coerce')

    # c) Optimize Categorical Columns
    categorical_cols = ['state', 'gender', 'education', 'policy_type', 'vehicle_class']
    for col in categorical_cols:
        if col in df.columns:
             df[col] = df[col].astype('category')
             

    # 3. Dealing with Null Values (Exercise 4)

    # a) Impute Numerical Nulls ('income'): using the median
    median_income = df['income'].median()
    df['income'].fillna(median_income, inplace=True)

    # b) Impute Categorical Nulls (Gender, Education, EmploymentStatus, Response): using the mode
    for col in ['gender', 'education', 'employmentstatus', 'response']:
        if col in df.columns and df[col].isnull().any():
            mode_val = df[col].mode()[0] 
            df[col].fillna(mode_val, inplace=True)
            
            
    # 4. Final Numeric Conversion (Exercise 4 - last step)

    numeric_cols_to_int = ['customer_lifetime_value', 'income', 'monthly_premium_auto', 
                           'total_claim_amount', 'months_since_last_claim', 
                           'months_since_policy_inception', 'number_of_policies']
                           
    for col in numeric_cols_to_int:
        if col in df.columns and df[col].dtype != 'category':
            df[col] = df[col].round(0).astype('Int64')

    
    # 5. Dealing with Duplicates (Exercise 5)
    
    # Drop duplicates, keeping the first occurrence
    df.drop_duplicates(keep='first', inplace=True)

    # Reset index
    df.reset_index(drop=True, inplace=True)
    
    return df